# 05 · Prepare Documents for RAG
**Goal:** load text, split it into overlapping chunks, attach source metadata, and inspect chunk quality before indexing.

RAG quality depends heavily on what is retrieved. Chunking is a design choice, not a universal constant.

In [ ]:
%pip install -q langchain-text-splitters pypdf

## Create a sample source document

In [ ]:
from pathlib import Path

docs_dir = Path("sample_docs")
docs_dir.mkdir(exist_ok=True)
sample = docs_dir / "student_services.txt"
sample.write_text(
    "Library Services\n"
    "Students can borrow books using their campus identity card. Renewals are available through the library portal.\n\n"
    "IT Support\n"
    "For account access problems, first check your username and keyboard layout. Use the password recovery workflow if needed.\n\n"
    "Sports Facilities\n"
    "Court bookings are made through the recreation desk. Users should cancel reservations they can no longer attend.\n",
    encoding="utf-8",
)
text = sample.read_text(encoding="utf-8")
print(text)

## Compare chunking approaches

In [ ]:
def fixed_chunks(text, size=180, overlap=30):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Require size > overlap >= 0")
    result, start = [], 0
    while start < len(text):
        result.append(text[start:start + size])
        start += size - overlap
    return result

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_fixed = fixed_chunks(text)
chunks_recursive = recursive_splitter.split_text(text)
print("Fixed-size chunks:", len(chunks_fixed))
print("Recursive chunks:", len(chunks_recursive))
for i, chunk in enumerate(chunks_recursive, start=1):
    print(f"\n--- chunk {i} ({len(chunk)} chars) ---\n{chunk}")

## Attach metadata to each chunk

In [ ]:
chunk_records = [
    {
        "id": f"{sample.stem}_{i:03d}",
        "text": chunk,
        "metadata": {"source": sample.name, "chunk_index": i},
    }
    for i, chunk in enumerate(chunks_recursive)
]
chunk_records[:2]

## Load a PDF you provide (optional)

In [ ]:
# Point this at a PDF file that you own or are allowed to process.
from pypdf import PdfReader

def read_pdf(path):
    reader = PdfReader(path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        if page_text.strip():
            pages.append({"page": page_number, "text": page_text})
    return pages

# Example:
# pages = read_pdf("sample_docs/your_document.pdf")
# print("Pages with text:", len(pages))
# print(pages[0] if pages else "No extractable text found.")

## Chunking checklist
- Does each chunk contain enough context to answer a likely question?
- Are headings kept with the paragraphs they describe?
- Is overlap large enough to preserve boundary context without creating too much duplication?
- Can every chunk be traced to its source and page/section?
- Have you inspected examples manually instead of choosing chunk size only by chunk count?

**Exercise:** try chunk sizes of 100, 250, and 400 characters and compare the resulting boundaries.